# 07 — Ruang Warna, Kolorimetri, dan Koreksi Pencahayaan

### Dari respons spektral sensor menuju pengukuran warna yang dapat dipertanggungjawabkan

**Seri:** Pengolahan Citra Digital dan *Scientific Imaging* · **Bahasa:** Indonesia · **Prasyarat:** representasi citra, statistik dasar, matriks, dan pemrograman Python.

Warna pada sebuah foto bukan sifat intrinsik piksel: nilai yang direkam dibentuk oleh **spektrum iluminan**, **reflektansi objek**, **respons spektral sensor**, **optika**, **penguatan elektronika**, dan **pemrosesan kamera**. Akibatnya, dua perangkat dapat merekam objek yang sama dengan RGB berbeda; bahkan satu perangkat dapat berubah ketika sumber cahaya diganti. Pada pengukuran warna, kesalahan memahami rantai akuisisi sering lebih serius daripada kesalahan memilih algoritma machine learning.

Bab ini bergerak dari fisika spektral menuju matematika transformasi warna, pengujian numerik, dan satu studi kasus utama: **koreksi warna pada inspeksi sampel menggunakan target referensi sintetis**, dengan pemisahan data kalibrasi, validasi, uji, serta *domain shift*. Semua data studi kasus dibangkitkan secara sintetis; tidak ada klaim bahwa angka simulasi merupakan akurasi kamera atau ColorChecker fisik.

> **Praktikum:** jalankan sel dari atas ke bawah. Semua hasil, termasuk angka evaluasi dan visualisasi, dihitung oleh Python, bukan tabel yang ditulis sebagai hasil eksperimen nyata.

## Peta kompetensi dan pertanyaan pemandu

Setelah mempelajari bab ini, pembaca diharapkan mampu:

1. Menurunkan hubungan antara iluminan, reflektansi, dan respons spektral sensor tiga kanal.
2. Membedakan **camera RGB linear**, **linear sRGB**, dan **sRGB terkode**; menerapkan fungsi transfer sRGB tanpa mencampurkannya dengan gamma generik.
3. Menjelaskan CIE XYZ, CIELAB, HSV, titik putih D65, dan mengapa $94 E_{00}$ tidak sah dihitung langsung dari RGB kamera.
4. Memformulasikan *white balance* diagonal, pemetaan warna linear $3\times3$, dan regularisasi dalam fitting matriks kalibrasi.
5. Mengevaluasi error warna, clipping, variasi iluminan, pengaruh koreksi terhadap klasifikasi, serta ketidakpastian antar-akuisisi.
6. Merancang validasi tanpa kebocoran data, menyatakan batas kesimpulan, dan membedakan kalibrasi visual dari metrologi warna.

**Pertanyaan diagnostik:** Jika satu patch hijau direkam di bawah lampu hangat, apakah penguatan kanal biru akan selalu mengembalikan nilai warna fisiknya? Mengapa dua gambar dengan rata-rata RGB sama bisa mempunyai warna lokal dan klasifikasi yang berbeda? Apakah hasil ΔE00 kecil pada data simulasi menjamin kamera sungguhan telah terkalibrasi?

**Alur:** fenomena fisik → model spektral → sistem encoding → ruang warna → koreksi → evaluasi dan ketidakpastian → studi kasus terpadu.

## 1. Kontrak numerik, terminologi, dan lingkungan

Kita gunakan vektor berurutan **R–G–B** dan angka floating-point dengan rentang nominal $[0,1]$. Namun **rentang sama tidak berarti makna sama**:

- **Camera RGB linear:** tiga keluaran sensor sesudah koreksi hitam, sebelum kurva nonlinier; bergantung kamera dan iluminan.
- **Linear sRGB:** koordinat warna sRGB yang telah melalui transformasi perangkat dan mempunyai respons linear terhadap intensitas relatif.
- **Encoded sRGB:** nilai untuk disimpan atau ditampilkan, sesudah fungsi transfer nonlinear sRGB.
- **CIE XYZ/CIELAB:** representasi kolorimetrik yang memerlukan asumsi *standard observer* dan *reference white*; di sini transformasi memakai D65, pengamat $2^\circ$, dan input sRGB yang benar.

**Kontrak:** fungsi `rgb2lab` menerima **sRGB yang telah dienkode dengan benar**, bukan angka sensor mentah. `ΔE00` dibandingkan pada pasangan Lab yang memiliki putih acuan sama. Setiap hasil simulator dilaporkan sebagai hasil sintetis; setiap operasi yang memakai *ground truth* pada proses fitting harus dibatasi pada data latih atau kalibrasi yang memang tersedia.

In [ ]:
import platform, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.integrate import trapezoid
from skimage import color
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

SEED = 20261112
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': False, 'font.size': 10})
print('Python:', platform.python_version(), '| NumPy:', np.__version__, '| scikit-image:', __import__('skimage').__version__)

## 2. Model pembentukan warna: reflektansi, iluminasi, dan sensor

Dalam model objek opak yang diamati pada geometri tetap, radiansi spektral yang mencapai sensor sering didekati sebagai hasil kali $E(\lambda)R(\lambda)$, dengan $E$ intensitas spektral iluminan dan $R$ reflektansi spektral efektif objek. Respons satu kanal sensor adalah

$$
y_c = t\,g_c\int_{\lambda_{\min}}^{\lambda_{\max}} E(\lambda)R(\lambda)S_c(\lambda)\,\mathrm{d}\lambda + b_c + \eta_c,\qquad c\in\{R,G,B\}.
$$

Di sini $t$ waktu integrasi, $g_c$ penguatan kanal, $S_c$ sensitivitas spektral sensor yang mencakup filter warna, $b_c$ offset, dan $\eta_c$ noise. Persamaan ini menyederhanakan geometri, *stray light*, respons nonlinier, iluminasi arah, dan interaksi subpermukaan; karena itu ia adalah model awal, bukan model lengkap seluruh kamera.

Poin utama: **warna sensor tidak ditentukan reflektansi saja**. Variasi lampu mengubah integralnya; perubahan matriks spektral sensor antarperangkat mengubah respons meskipun lampu dan objek sama.

In [ ]:
# Ilustrasi SPEKTRAL sintetis, bukan respons CIE 1931 dan bukan kurva sensor terukur.
lam = np.arange(380., 781., 5.)
def gauss_spec(mu,sigma): return np.exp(-.5*((lam-mu)/sigma)**2)
S=np.column_stack([gauss_spec(610,52),gauss_spec(540,48),gauss_spec(460,42)])
S/=np.max(S,axis=0)
E_day=.65+.30*gauss_spec(465,95)+.12*gauss_spec(600,130)
E_warm=.32+.85*gauss_spec(620,95)
R1=.12+.77*gauss_spec(530,70)
R2=.15+.62*gauss_spec(620,68)
def simulated_sensor_response(E,R):
    q=trapezoid(E[:,None]*R[:,None]*S,lam,axis=0)
    return q / trapezoid(S,lam,axis=0)
fig,ax=plt.subplots(1,3,figsize=(12,3.1))
ax[0].plot(lam,S);ax[0].set_title('Sensitivitas sensor (model)');ax[0].legend(['R','G','B'])
ax[1].plot(lam,E_day,label='Iluminan A (sintetis)');ax[1].plot(lam,E_warm,label='Iluminan B (sintetis)');ax[1].legend();ax[1].set_title('Spektrum iluminan')
ax[2].plot(lam,R1,label='Material 1');ax[2].plot(lam,R2,label='Material 2');ax[2].legend();ax[2].set_title('Reflektansi dua material')
for a in ax: a.set(xlabel='Panjang gelombang (nm)',ylabel='Nilai relatif');a.grid(alpha=.2)
plt.tight_layout();plt.show()
for i,R in enumerate([R1,R2],start=1):
    print(f'Material {i}: respons E_A = {simulated_sensor_response(E_day,R).round(3)}; respons E_B = {simulated_sensor_response(E_warm,R).round(3)}')

## 3. Trikromasi dan batas identifikasi spektrum

Satu spektrum kontinu $R(\lambda)$ mempunyai banyak derajat kebebasan, tetapi kamera RGB biasa hanya menghasilkan tiga angka. Pemetaan dari fungsi spektral ke tiga komponen **tidak injektif**: beberapa spektrum dapat menghasilkan RGB identik atau sangat dekat pada satu sensor/iluminan, walau berbeda pada iluminan lain (*metamerism*). Masalah ini menjelaskan mengapa pemulihan reflektansi spektral dari satu foto RGB secara umum merupakan *inverse problem* dengan solusi tidak unik.

Bahkan apabila sensor terkalibrasi baik, **white balance tidak dapat memulihkan semua informasi spektral yang tidak direkam**. Bila targetnya adalah spektrofotometri, kamera RGB harus divalidasi untuk rentang material dan kondisi akuisisi tertentu atau diganti dengan sensor multispektral.

In [ ]:
# Konstruksi matematis dua reflektansi berbeda dengan respons sangat mirip
# pada satu iluminan: proyeksikan gangguan acak ke null space sistem 3 kanal.
weights=np.full_like(lam,5.0);weights[0]=weights[-1]=2.5
A=(S*E_day[:,None])*weights[:,None] # bobot trapezoid sesuai integral numerik
base=.48+.08*np.cos((lam-550)/105)
rr=np.random.default_rng(SEED+1)
v=rr.normal(size=len(lam))
# proyeksi ke ruang nol baris A.T
proj = A@np.linalg.solve(A.T@A, A.T@v)
null=v-proj
null/=np.max(np.abs(null))
R_a=np.clip(base+.18*null,0,1)
R_b=np.clip(base-.18*null,0,1)
y_Aa=simulated_sensor_response(E_day,R_a);y_Ab=simulated_sensor_response(E_day,R_b)
y_Ba=simulated_sensor_response(E_warm,R_a);y_Bb=simulated_sensor_response(E_warm,R_b)
fig,ax=plt.subplots(1,2,figsize=(10,3))
ax[0].plot(lam,R_a,label='Reflektansi A');ax[0].plot(lam,R_b,label='Reflektansi B');ax[0].legend();ax[0].set(xlabel='nm',ylabel='Reflektansi',title='Dua spektrum berbeda')
ax[1].bar(np.arange(3)-0.2,y_Aa,width=.4,label='A pada iluminan A');ax[1].bar(np.arange(3)+0.2,y_Ab,width=.4,label='B pada iluminan A');ax[1].set_xticks(range(3),['R','G','B']);ax[1].legend();ax[1].set_title('Respons identik, kondisi tertentu')
plt.tight_layout();plt.show()
print('Jarak respons pada iluminan A:',np.linalg.norm(y_Aa-y_Ab))
print('Jarak respons pada iluminan B:',np.linalg.norm(y_Ba-y_Bb))
assert np.linalg.norm(y_Aa-y_Ab)<1e-8

## 4. Fungsi transfer sRGB: pengkodean bukan pengukuran linear

Bila $C_\mathrm{lin}$ adalah satu komponen **linear sRGB** pada $[0,1]$, fungsi transfer sRGB (*encoding*) adalah

$$
C_\mathrm{enc}=\begin{cases}
12.92\,C_\mathrm{lin}, & C_\mathrm{lin}\leq0.0031308,\\
1.055\,C_\mathrm{lin}^{1/2.4}-0.055, & C_\mathrm{lin}>0.0031308.
\end{cases}
$$

Inversnya memakai batas $0.04045$:

$$
C_\mathrm{lin}=\begin{cases}
C_\mathrm{enc}/12.92, & C_\mathrm{enc}\leq0.04045,\\
[(C_\mathrm{enc}+0.055)/1.055]^{2.4}, & C_\mathrm{enc}>0.04045.
\end{cases}
$$

Fungsi ini **bukan sekadar menaikkan pangkat dengan satu angka gamma**. Rata-rata dua nilai encoded sRGB tidak sama dengan rata-rata intensitas linear. Kalibrasi sensor, koreksi gain radiometrik, penjumlahan cahaya, dan model shot noise lazimnya lebih sah dibahas di domain linear. Pemrosesan tampilan dapat dilakukan di domain sRGB encoded bila memang itulah definisi tugasnya.

In [ ]:
def lin_to_srgb(x):
    x=np.clip(np.asarray(x,dtype=float),0,1)
    return np.where(x<=.0031308,12.92*x,1.055*np.power(x,1/2.4)-.055)
def srgb_to_lin(x):
    x=np.clip(np.asarray(x,dtype=float),0,1)
    return np.where(x<=.04045,x/12.92,np.power((x+.055)/1.055,2.4))
zz=np.linspace(0,1,1001)
assert np.max(np.abs(srgb_to_lin(lin_to_srgb(zz))-zz))<1e-12
fig,ax=plt.subplots(1,2,figsize=(10,3.3))
ax[0].plot(zz,lin_to_srgb(zz),label='sRGB encoding');ax[0].plot(zz,zz,'--',label='identitas');ax[0].legend();ax[0].set(xlabel='Intensitas linear',ylabel='Nilai encoded',title='Fungsi transfer sRGB')
bar=np.array([.1,.9]);avg_lin=bar.mean();avg_encoded=lin_to_srgb(bar).mean()
ax[1].bar(['Rerata linear','Decode(rerata encoded)'],[avg_lin,srgb_to_lin(avg_encoded)]);ax[1].set_ylim(0,1);ax[1].set_title('Mengapa rerata encoded tidak linear')
plt.tight_layout();plt.show()
print('Rata-rata linear=',avg_lin,' | Rata-rata setelah jalur encoded=',srgb_to_lin(avg_encoded))

## 5. Dari linear sRGB ke CIE XYZ: matriks, titik putih, dan batas gamut

Pada standar sRGB dengan titik putih D65, transformasi relatif dari **linear sRGB** menjadi CIE XYZ menggunakan matriks $\mathbf M$:

$$
\begin{bmatrix}X\\Y\\Z\end{bmatrix}=
\begin{bmatrix}
0.4124564&0.3575761&0.1804375\\
0.2126729&0.7151522&0.0721750\\
0.0193339&0.1191920&0.9503041
\end{bmatrix}
\begin{bmatrix}R_\mathrm{lin}\\G_\mathrm{lin}\\B_\mathrm{lin}\end{bmatrix}.
$$

Komponen $Y$ merupakan *relative luminance* untuk kondisi dan satuan normalisasi yang ditetapkan, **bukan langsung luminansi absolut dalam cd/m²**. XYZ dapat memuat warna di luar gamut sRGB. Mengubah hasil matriks di luar rentang menjadi RGB tampilan dan melakukan *clipping* akan kehilangan informasi. Perangkat non-sRGB harus diubah melalui profil atau matriks karakterisasi yang sesuai sebelum memakai transformasi ini.

In [ ]:
M_SRGB_TO_XYZ=np.array([[.4124564,.3575761,.1804375],[.2126729,.7151522,.0721750],[.0193339,.1191920,.9503041]])
primaries=np.eye(3)
manual=primaries@M_SRGB_TO_XYZ.T
library=color.rgb2xyz(primaries.reshape((3,1,3))).reshape(3,3)
print('XYZ primer secara manual:\n',manual)
print('XYZ primer dari scikit-image:\n',library)
print('Selisih maksimum:',np.abs(manual-library).max())
assert np.max(np.abs(manual-library))<5e-4

## 6. HSV, CIELAB, dan fungsi yang tepat untuk tujuan berbeda

**HSV** memetakan nilai RGB menjadi *hue, saturation,* dan *value*. Representasi ini membantu pengaturan warna tampilan dan pemilihan wilayah warna, tetapi $H$ menjadi tidak stabil ketika saturasi mendekati nol; jarak Euclidean di HSV juga bukan metrik beda warna perseptual yang baku.

**CIELAB** memetakan tristimulus XYZ relatif terhadap putih acuan $(X_n,Y_n,Z_n)$. Dengan $\delta=6/29$ dan

$$
f(t)=\begin{cases}t^{1/3}, &t>\delta^3,\\ t/(3\delta^2)+4/29,&t\leq\delta^3,\end{cases}
$$

komponennya adalah

$$
L^*=116f(Y/Y_n)-16,\quad a^*=500[f(X/X_n)-f(Y/Y_n)],\quad b^*=200[f(Y/Y_n)-f(Z/Z_n)].
$$

Nilai $L^*$ menggambarkan *lightness* dan tidak identik dengan luminansi $Y$. CIELAB lebih cocok untuk pembandingan kolorimetrik daripada RGB mentah, tetapi tetap memerlukan titik putih dan tidak seragam secara perseptual di semua wilayah gamut. Kalibrasi sRGB yang benar harus mendahului `rgb2lab`.

In [ ]:
# Palette demo: Hue pada warna jenuh serta ketidakstabilan Hue dekat abu-abu.
sat=np.array([[[.9,.1,.2],[.1,.8,.2],[.15,.25,.9],[.5,.5,.5],[.5001,.4999,.5]]])
hsv=color.rgb2hsv(sat)[0]
lab=color.rgb2lab(sat)[0]
fig,ax=plt.subplots(1,2,figsize=(10,2.8))
ax[0].imshow(sat,aspect='auto');ax[0].set_title('Lima contoh sRGB encoded');ax[0].set_xticks(range(5));ax[0].set_yticks([])
ax[1].bar(range(5),hsv[:,0]);ax[1].set_ylim(0,1);ax[1].set_title('Hue HSV (0–1)');ax[1].set_xlabel('Indeks patch');
plt.tight_layout();plt.show()
print(pd.DataFrame({'H':hsv[:,0],'S':hsv[:,1],'L*':lab[:,0],'a*':lab[:,1],'b*':lab[:,2]}).round(3))

## 7. Beda warna: $\Delta E_{ab}^*$ dan $\Delta E_{00}$

Jarak Euclidean CIELAB klasik didefinisikan sebagai

$$
\Delta E_{ab}^*=\sqrt{(\Delta L^*)^2+(\Delta a^*)^2+(\Delta b^*)^2}.
$$

Karena jarak CIELAB tidak selalu sesuai beda perseptual, CIEDE2000 memperkenalkan koreksi pada *lightness*, *chroma*, *hue*, dan interaksinya:

$$
\Delta E_{00}=\sqrt{\left(\frac{\Delta L'}{k_LS_L}\right)^2+
\left(\frac{\Delta C'}{k_CS_C}\right)^2+
\left(\frac{\Delta H'}{k_HS_H}\right)^2+
R_T\left(\frac{\Delta C'}{k_CS_C}\right)\left(\frac{\Delta H'}{k_HS_H}\right)}.
$$

Dalam studi ini digunakan $k_L=k_C=k_H=1$ sebagai kondisi referensi grafis; konteks industri dapat memerlukan parameter yang berbeda. Implementasi rumus penuh memuat cabang hue dan kasus diskontinu, sehingga kita memakai `skimage.color.deltaE_ciede2000` yang teruji, lalu memeriksa satu pasangan rujukan numerik dari literatur (Sharma, Wu, dan Dalal, 2005). Tidak ada ambang ΔE00 yang universal untuk semua material, layar, dan prosedur pengamatan.

In [ ]:
L1=np.array([50.0000,2.6772,-79.7751])
L2=np.array([50.0000,0.0000,-82.7485])
de00=float(color.deltaE_ciede2000(L1,L2))
de76=float(color.deltaE_cie76(L1,L2))
print('Kasus validasi CIEDE2000: ΔE00=',round(de00,5),'(rujukan 2.0425)')
print('Pada pasangan sama: ΔE76=',round(de76,5))
assert abs(de00-2.0425)<5e-4

## 8. White balance tidak sama dengan kalibrasi warna

Model diagonal *white balance* pada nilai linear dapat ditulis

$$
\hat{\mathbf r}_\mathrm{WB}=\mathbf D\,\mathbf y,\qquad
\mathbf D=\operatorname{diag}(d_R,d_G,d_B).
$$

Jika patch netral dengan target linear $\mathbf r_n$ tersedia pada setiap akuisisi dan respons kamera yang terukur adalah $\mathbf y_n$, maka pendekatan sederhana adalah $d_c=r_{n,c}/y_{n,c}$. Perkalian diagonal dapat mengoreksi penguatan kanal di dekat titik netral, tetapi tidak memperbaiki *channel crosstalk*, respons spektral yang berbeda, nonlinieritas, *spatial shading*, dan clipping.

**Gray-world** memperkirakan bahwa rerata adegan seharusnya netral. Asumsi ini dapat gagal total untuk pemandangan dominan hijau, kulit, atau objek berwarna tunggal. Di bawah ini, pendekatan *gray-world* ditunjukkan sebagai diagnostik, bukan standar kalibrasi kolorimetrik.

In [ ]:
# Gray-world failure: dua adegan dengan iluminan identik tetapi distribusi warna objek berbeda.
scene_a=np.tile(np.array([.32,.32,.32]),(80,120,1))
scene_b=np.tile(np.array([.17,.57,.21]),(80,120,1))
ill=np.array([1.18,.94,.78])
def gray_world_gain(raw):
    mu=raw.mean(axis=(0,1));return mu.mean()/np.maximum(mu,1e-9)
g1=gray_world_gain(scene_a*ill);g2=gray_world_gain(scene_b*ill)
print('Gain gray-world pada adegan netral:',g1.round(3))
print('Gain gray-world pada adegan dominan hijau:',g2.round(3))
fig,ax=plt.subplots(1,2,figsize=(8,2.4))
ax[0].imshow(np.clip(scene_a*ill*g1,0,1));ax[0].set_title('Adegan netral: koreksi asumsi GW')
ax[1].imshow(np.clip(scene_b*ill*g2,0,1));ax[1].set_title('Adegan hijau: "dinetralkan" keliru')
for a in ax:a.axis('off')
plt.tight_layout();plt.show()

## 9. Matriks koreksi warna dan alasan regularisasi

Secara lokal, jika nilai sudah linear dan rentang eksposur tidak melampaui batas, pemetaan kamera ke linear sRGB dapat didekati oleh matriks $3\times3$:

$$
\widehat{\mathbf r}_i=\mathbf W^{\mathsf T}\mathbf y_i,
\qquad \mathbf W\in\mathbb R^{3\times 3}.
$$

Dengan menyusun pengukuran patch ke matriks $\mathbf X\in\mathbb R^{N\times3}$ dan referensi ke $\mathbf Y\in\mathbb R^{N\times3}$, kita minimalkan penalti kuadrat dengan regularisasi ridge:

$$
\mathbf W^*=\arg\min_\mathbf W\;\|\mathbf X\mathbf W-\mathbf Y\|_F^2+\lambda\|\mathbf W\|_F^2,
\qquad
\mathbf W^*=(\mathbf X^{\mathsf T}\mathbf X+\lambda\mathbf I)^{-1}\mathbf X^{\mathsf T}\mathbf Y.
$$

Penalti $\lambda$ menurunkan ketidakstabilan matriks ketika patch terlalu mirip atau noise besar. Jika respons kamera mengalami saturasi, tidak ada matriks linear yang dapat memulihkan sinyal yang hilang. Model matriks juga hanya memadai sebagai pendekatan jika hubungan warna perangkat dan iluminan pada domain operasi cukup linier.

## 10. Desain studi kasus: target multiwarna dengan referensi diketahui

Kita membuat **target 24 patch sintetis** yang sengaja dibagi ke empat kelas warna: netral, hangat, hijau, dan dingin. Warna referensi disimpan sebagai nilai sRGB D65 **yang ditentukan oleh penulis simulator**, bukan hasil pengukuran standar ColorChecker/Macbeth. Dengan demikian, ΔE00 merupakan galat terhadap target sintetik, bukan terhadap sertifikat kolorimetri.

**Protokol:** pada tiap kelas, empat warna disediakan untuk kalibrasi, satu untuk validasi model, dan satu *warna yang belum dilihat* untuk pengujian. Selain pemisahan warna, indeks akuisisi juga dipisah: akuisisi train tidak digunakan dalam validasi atau pengujian. Di setiap akuisisi terdapat patch netral referensi yang **secara eksplisit diizinkan** untuk estimasi white balance; ini mensimulasikan protokol fisik di mana target netral berada dalam frame. Tidak satu pun nilai referensi pada warna uji dipakai untuk fitting matriks.

In [ ]:
CLASS_NAMES=np.array(['Netral','Hangat','Hijau','Dingin'])
REF_SRGB=np.array([
    [.53,.53,.53],[.13,.13,.13],[.84,.84,.84],[.34,.34,.34],[.68,.68,.68],[.43,.43,.43],
    [.87,.30,.22],[.79,.50,.18],[.96,.64,.27],[.59,.20,.13],[.93,.46,.43],[.71,.30,.24],
    [.24,.63,.31],[.18,.77,.48],[.61,.82,.26],[.23,.40,.18],[.22,.68,.58],[.45,.70,.23],
    [.16,.43,.77],[.31,.20,.74],[.31,.73,.89],[.18,.25,.54],[.66,.40,.80],[.37,.54,.84],
],dtype=float)
REF_LIN=srgb_to_lin(REF_SRGB)
LABELS=np.repeat(np.arange(4),6)
CAL_IDX=np.array([i for c in range(4) for i in range(6*c,6*c+4)],dtype=int)
VAL_IDX=np.array([4,10,16,22]);TEST_IDX=np.array([5,11,17,23])
assert len(set(CAL_IDX)&set(VAL_IDX))==len(set(CAL_IDX)&set(TEST_IDX))==len(set(VAL_IDX)&set(TEST_IDX))==0
assert set(np.r_[CAL_IDX,VAL_IDX,TEST_IDX])==set(range(24))
fig,ax=plt.subplots(figsize=(11,3.1))
ax.imshow(REF_SRGB.reshape(4,6,3),interpolation='nearest')
ax.set_xticks(range(6),[f'P{j+1}' for j in range(6)]);ax.set_yticks(range(4),CLASS_NAMES)
for row in range(4):
    for col in range(6):
        role='C' if row*6+col in CAL_IDX else 'V' if row*6+col in VAL_IDX else 'U'
        ax.text(col,row,role,ha='center',va='center',color='black',fontweight='bold',bbox=dict(fc='white',alpha=.8,pad=.8,ec='none'))
ax.set_title('Target sintetis: C=kalibrasi, V=validasi, U=uji; bukan ColorChecker standar')
plt.tight_layout();plt.show()

## 11. Digital twin kamera: ilustrasi yang eksplisit menyatakan asumsi

Dalam studi ini, representasi setiap patch adalah intensitas linear ideal $\mathbf r_i$; kamera simulasi menghasilkan

$$
\mathbf y_{i,s}=\operatorname{clip}_{[0,1]}\!\left[\alpha_s\,q_{i,s}\;\mathbf M_s\,\operatorname{diag}(\boldsymbol\ell_s)\mathbf r_i+\boldsymbol\eta_{i,s}\right].
$$

Indeks $i$ menunjukkan patch dan $s$ menunjukkan akuisisi. Besaran $\alpha_s$ adalah eksposur, $q_{i,s}$ faktor shading spasial, $\mathbf M_s$ matriks pencampuran kanal sintetik, dan $\boldsymbol\ell_s$ gain iluminan efektif. Noise menggunakan pendekatan foton Poisson dan noise baca Gaussian. **Model ini bukan simulator spektral kamera yang teridentifikasi dari pengukuran nyata**; eksposur, gain dan matriks tidak diberi satuan radiometrik mutlak.

*Domain stress* mengganti iluminan dan matriks respons sensor sekaligus: ini meniru perubahan perangkat/lampu yang tidak sepenuhnya dapat diatasi oleh koreksi tunggal. Seluruh bilangan acak diturunkan dari seed tersendiri per akuisisi, sehingga hasil dapat direproduksi tanpa internet.

In [ ]:
CAMERA_M=np.array([[1.06,.065,.018],[.038,.91,.052],[.024,.085,.955]])
STRESS_M=np.array([[.91,.095,.045],[.065,1.05,.023],[.055,.11,.86]])

# Hanya skenario nominal train, val, test yang berbagi keluarga distribusi.
SPLIT_SEEDS={'train':np.arange(100,118),'val':np.arange(200,208),
             'test':np.arange(300,314),'stress':np.arange(400,408)}

def simulate_acquisition(seed,split):
    rg=np.random.default_rng(int(seed)+SEED)
    if split=='stress':
        illuminant=np.array([.81,.96,1.28])*rg.lognormal(mean=0,sigma=.045,size=3)
        M=STRESS_M
    else:
        illuminant=np.array([1.18,.96,.83])*rg.lognormal(mean=0,sigma=.065,size=3)
        M=CAMERA_M
    exposure=float(rg.uniform(.83,.95))
    x=(np.arange(24)%6)/5
    y=(np.arange(24)//6)/3
    shading=.88+.10*x+.05*y
    expected=(REF_LIN*illuminant)@M.T * (exposure*shading[:,None])
    # Poisson dan read noise pada kuantitas ternormalisasi; bukan kalibrasi elektron sensor.
    electrons=8500.
    noisy=rg.poisson(np.maximum(expected,0)*electrons)/electrons + rg.normal(0,.0032,size=expected.shape)
    meas=np.clip(noisy,0,1)
    return {'id':int(seed),'split':split,'measured':meas,'ideal':REF_LIN.copy(),
            'exposure':exposure,'illuminant':illuminant,'shading':shading,
            'clipped_sensor':int(np.count_nonzero(noisy>=1))}

COLLECTION={key:[simulate_acquisition(s,key) for s in values] for key,values in SPLIT_SEEDS.items()}
assert len(set(SPLIT_SEEDS['train'])&set(SPLIT_SEEDS['test']))==0
print('Jumlah akuisisi:',{k:len(v) for k,v in COLLECTION.items()})
print('Contoh gain iluminan:',COLLECTION['train'][0]['illuminant'].round(3))

## 12. Menampilkan citra kamera tidak sama dengan merender sRGB terkalibrasi

Representasi diagram berikut sengaja menganggap angka kamera mentah sebagai linear sRGB untuk keperluan **pratinjau visual**, sehingga kita dapat melihat bagaimana bias sensor dan iluminan tampak di layar. Ini **bukan proses warna kamera yang sah** dan tidak boleh dipakai sebagai konversi fisik tanpa pemetaan perangkat.

Perhatikan perbedaan antara target ideal, angka sensor yang dipaksakan ke tampilan, serta iluminan domain stress. Identitas RGB angka ketiga gambar tidak dapat dinilai secara andal dari tampilan layar saja; evaluasi numerik diperlukan.

In [ ]:
def swatch_show(a):return lin_to_srgb(a).reshape(4,6,3)
a=COLLECTION['test'][0];b=COLLECTION['stress'][0]
fig,ax=plt.subplots(1,3,figsize=(12,3.2))
for k,(im,title) in enumerate([(REF_SRGB.reshape(4,6,3),'Target sintetik sRGB'),
                               (swatch_show(a['measured']),'Kamera: test (pratinjau naif)'),
                               (swatch_show(b['measured']),'Kamera: stress (pratinjau naif)')]):
    ax[k].imshow(im);ax[k].set_title(title);ax[k].set_xticks([]);ax[k].set_yticks([])
plt.tight_layout();plt.show()

## 13. Implementasi white balance berbasis patch netral terukur

Untuk setiap akuisisi kita izinkan patch referensi pertama (indeks 0) menjadi acuan netral. Gain diagonal dihitung dari **nilai referensi patch netral yang tersedia sebagai instrumen kalibrasi**, bukan dari label warna uji. Bagi foto operasional tanpa referensi netral, strategi ini **tidak dapat langsung dipakai**; diperlukan perkiraan iluminan atau protokol akuisisi berbeda.

Hindari membagi angka sensor yang mendekati nol tanpa perlindungan numerik. Evaluasi clipping dilaksanakan **sebelum** pemotongan rentang untuk display, agar saturasi tidak tersembunyi.

In [ ]:
def white_balance(d):
    gain=REF_LIN[0]/np.maximum(d['measured'][0],.005)
    return d['measured']*gain

d=COLLECTION['test'][0]
wb=white_balance(d)
print('Gain WB dari patch netral:',(REF_LIN[0]/d['measured'][0]).round(4))
print('Patch netral sesudah WB:',wb[0].round(5),' | nilai rujukan:',REF_LIN[0].round(5))
assert np.max(np.abs(wb[0]-REF_LIN[0]))<1e-10
fig,ax=plt.subplots(1,3,figsize=(11,3))
for a0,im,t in zip(ax,[REF_SRGB.reshape(4,6,3),swatch_show(d['measured']),swatch_show(wb)],
                   ['Referensi','Kamera mentah (ditampilkan)','White balance']):
    a0.imshow(im);a0.set_title(t);a0.axis('off')
plt.tight_layout();plt.show()

## 14. Fitting *color correction matrix* (CCM) hanya pada split train

Matriks $3\times3$ untuk mengoreksi seluruh patch seharusnya diperoleh dari warna kalibrasi dan akuisisi train saja. Kita bandingkan dua cara:

1. **CCM:** matriks global dipasang langsung dari camera RGB linear ke linear sRGB.
2. **WB + CCM:** setiap frame terlebih dahulu di-*white balance* menggunakan patch netral yang tersedia, kemudian matriks global dipasang dari pengamatan latih.

Keduanya merupakan model matematis sederhana. *WB + CCM* dapat lebih tahan terhadap variasi gain kanal antar-akuisisi, tetapi masih mungkin gagal di domain stress akibat perubahan matriks spektral atau nonlinieritas. Model tanpa intersep menjaga titik nol dipetakan ke nol; keterbatasan ini disengaja dan dibahas ketika membaca hasil.

In [ ]:
def fit_matrix(train_acquisitions,reg=0.0,use_wb=False):
    X=np.concatenate([(white_balance(d) if use_wb else d['measured'])[CAL_IDX] for d in train_acquisitions])
    Y=np.concatenate([d['ideal'][CAL_IDX] for d in train_acquisitions])
    W=np.linalg.solve(X.T@X+reg*np.eye(3),X.T@Y)
    return W

def predict_linear(d,method,models=None):
    raw=d['measured']
    if method=='Tanpa koreksi':return raw
    if method=='White balance':return white_balance(d)
    if method=='CCM':return raw@models['CCM']
    if method=='WB + CCM':return white_balance(d)@models['WB + CCM']
    raise ValueError('Metode tidak dikenali: '+str(method))

METHODS=['Tanpa koreksi','White balance','CCM','WB + CCM']
print('CCM contoh λ=0.001 tanpa WB:\n',fit_matrix(COLLECTION['train'],reg=.001).round(3))
print('CCM contoh λ=0.001 setelah WB:\n',fit_matrix(COLLECTION['train'],reg=.001,use_wb=True).round(3))

## 15. Seleksi regularisasi: validasi, bukan test

Hyperparameter $\lambda$ dipilih dari grid terbatas melalui galat ΔE00 pada **warna validasi** dan **akuisisi validasi**. Sekali nilai dipilih, parameter dibekukan. Test dan stress tidak disentuh dalam proses seleksi. Dengan demikian estimasi generalisasi lebih jujur dibanding mencoba banyak parameter di test kemudian melaporkan yang terbaik.

Koreksi matriks dilakukan pada *linear RGB*. Untuk membandingkan hasil dengan referensi sRGB, hasilnya dibatasi ke $[0,1]$, dikodekan memakai sRGB transfer function, diubah ke Lab D65, kemudian dihitung ΔE00. Pelanggaran gamut dihitung secara terpisah sebelum *clipping*; nilai ΔE00 sesudah clipping tidak mengukur seluruh informasi warna yang sudah hilang.

In [ ]:
def rgb_to_lab_rows(rgb):
    arr=np.asarray(rgb,float).reshape(-1,1,3)
    return color.rgb2lab(arr,illuminant='D65',observer='2').reshape(-1,3)

def delta_e_rows(linear_pred,idx):
    lab_est=rgb_to_lab_rows(lin_to_srgb(linear_pred[idx]))
    lab_ref=rgb_to_lab_rows(REF_SRGB[idx])
    return color.deltaE_ciede2000(lab_est,lab_ref)

LAMBDA_GRID=np.array([0.,1e-5,1e-4,1e-3,1e-2,.1])
val_rows=[]
for use_wb,method in [(False,'CCM'),(True,'WB + CCM')]:
    for reg in LAMBDA_GRID:
        W=fit_matrix(COLLECTION['train'],reg=reg,use_wb=use_wb)
        ds=[delta_e_rows((white_balance(d) if use_wb else d['measured'])@W,VAL_IDX).mean() for d in COLLECTION['val']]
        val_rows.append({'method':method,'lambda':reg,'mean_dE00':np.mean(ds)})
selection=pd.DataFrame(val_rows)
best=selection.loc[selection.groupby('method')['mean_dE00'].idxmin()].copy().sort_values('method')
MODELS={row['method']:fit_matrix(COLLECTION['train'],float(row['lambda']),row['method']=='WB + CCM')
        for _,row in best.iterrows()}
print('Nilai λ terpilih hanya dengan validasi:')
print(best.to_string(index=False,float_format=lambda x:f'{x:.5f}'))
assert np.all(np.isfinite(np.concatenate([m.ravel() for m in MODELS.values()])))

## 16. Menguji pemetaan warna pada patch yang tidak ikut fitting

Pengujian memakai **empat patch uji** (masing-masing satu dari setiap kelas warna) dari **14 akuisisi test** yang belum digunakan pada fitting atau seleksi hyperparameter. Unit observasi utama adalah **akuisisi**, bukan sekadar 56 angka patch yang diasumsikan independen. Performa dievaluasi dengan rerata dan median ΔE00, persentil 90, serta rasio clipping sebelum encoding.

Hal yang dinilai ialah **kedekatan ke referensi sintetis**. Karena semua patch dan respons kamera didesain sendiri, hasil tidak dapat digunakan sebagai jaminan akurasi ColorChecker nyata atau tingkat penerimaan industri.

In [ ]:
def score_collection(collection,idx=TEST_IDX):
    out=[]
    for d in collection:
        for method in METHODS:
            est=predict_linear(d,method,MODELS)
            de=delta_e_rows(est,idx)
            for j,k in enumerate(idx):
                out.append({'id':d['id'],'split':d['split'],'method':method,
                            'patch':int(k),'kelas':CLASS_NAMES[LABELS[k]],'dE00':float(de[j]),
                            'clipped_fraction':float(np.mean((est[k]<0)|(est[k]>1)))})
    return pd.DataFrame(out)
res_test=score_collection(COLLECTION['test'])
summary=res_test.groupby('method',sort=False).agg(mean_dE00=('dE00','mean'),median_dE00=('dE00','median'),
                                                 p90_dE00=('dE00',lambda x:np.quantile(x,.9)),
                                                 clipping=('clipped_fraction','mean')).reindex(METHODS)
print(summary.round(3).to_string())
assert res_test.groupby('method').size().eq(len(COLLECTION['test'])*len(TEST_IDX)).all()

## 17. Audit interpretasi: distribusi, bukan hanya satu skor

Rata-rata ΔE00 merangkum hasil, tetapi bisa menyembunyikan patch tertentu yang gagal atau degradasi yang hanya muncul di beberapa akuisisi. Oleh sebab itu grafik per akuisisi penting untuk mengungkap ketidakstabilan yang tidak tampak pada tabel agregat. Pada studi sensor, pelaporan tiap kelompok warna atau rentang luminansi lebih informatif dibanding hanya skor global.

Ketika memilih metode untuk tugas tertentu, target praktis juga dapat berbeda: inspeksi untuk klasifikasi boleh toleran terhadap galat warna tertentu, sedangkan pengukuran kolorimetrik memerlukan validasi ΔE00 terhadap instrumen rujukan serta kondisi pengamatan baku.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,3.8))
res_test.boxplot(column='dE00',by='method',ax=ax[0],rot=20,grid=False)
ax[0].set_title('Sebaran ΔE00 patch uji');ax[0].set_xlabel('Metode');ax[0].set_ylabel('ΔE00')
ax[0].get_figure().suptitle('')
by_acq=res_test.groupby(['id','method'])['dE00'].mean().unstack()
for method in METHODS:ax[1].plot(by_acq.index,by_acq[method],marker='o',label=method,alpha=.8)
ax[1].set(xlabel='ID akuisisi test',ylabel='Mean ΔE00 per akuisisi',title='Variasi antar-akuisisi')
ax[1].legend(fontsize=8);ax[1].grid(alpha=.2)
plt.tight_layout();plt.show()

## 18. Mengapa rendering visual dan evaluasi kolorimetrik tidak identik?

Dalam pemrosesan tampilan, suatu koreksi dapat menghasilkan gambar yang tampak lebih netral meskipun tidak mendekati warna standar secara kuantitatif. Hal ini antara lain karena adaptasi visual pengamat, gamut display, dan kurva transfer. Untuk penelitian, tampilkan panel contoh **dan** ukur galat terhadap referensi.

Empat panel berikut menunjukkan patch sintetis uji pada akuisisi yang sama. Panel harus dibaca sebagai ilustrasi transformasi dari angka simulasi ke sRGB, bukan sebagai dokumentasi standar pencahayaan ruang laboratorium.

In [ ]:
d=COLLECTION['test'][4]
fig,ax=plt.subplots(1,5,figsize=(14,3.2))
ax[0].imshow(REF_SRGB.reshape(4,6,3));ax[0].set_title('Referensi')
for a,method in zip(ax[1:],METHODS):
    a.imshow(swatch_show(predict_linear(d,method,MODELS)));a.set_title(method,fontsize=9)
for a in ax:a.axis('off')
plt.tight_layout();plt.show()

## 19. Klasifikasi berbasis citra: apakah koreksi selalu meningkatkan F1?

Uji tambahan ini menggunakan klasifikasi empat jenis warna: **netral, hangat, hijau, dingin**. Model yang digunakan adalah regresi logistik multinomial dari `scikit-learn`; meskipun sederhana, ia sudah memungkinkan eksperimen terkontrol tentang *domain shift* dan preprocessing.

Agar perbandingan adil, **setiap metode menerima pembagian akuisisi dan warna yang sama**. `StandardScaler` dan bobot classifier hanya di-fit pada data train yang telah dikoreksi dengan metode masing-masing. Target uji adalah patch indeks `TEST_IDX` yang tidak pernah masuk train maupun validasi. Pembanding ini bukan evaluasi identitas warna tak terbatas; kelas dibuat berdasarkan palet sintetik, sehingga pengaruh koreksi bisa berbeda pada dataset lain.

In [ ]:
MODELS_ML={}
for method in METHODS:
    Xtr=np.concatenate([lin_to_srgb(predict_linear(d,method,MODELS))[CAL_IDX] for d in COLLECTION['train']])
    ytr=np.tile(LABELS[CAL_IDX],len(COLLECTION['train']))
    estimator=make_pipeline(StandardScaler(),LogisticRegression(C=3.0,max_iter=1000,random_state=SEED))
    estimator.fit(Xtr,ytr)
    MODELS_ML[method]=estimator

def evaluate_ml(collection,idx=TEST_IDX):
    rows=[]
    for method,model in MODELS_ML.items():
        for d in collection:
            X=lin_to_srgb(predict_linear(d,method,MODELS))[idx]
            yp=model.predict(X)
            yt=LABELS[idx]
            rows.extend([{'method':method,'split':d['split'],'id':d['id'],'patch':int(i),
                          'y_true':int(t),'y_pred':int(p)} for i,t,p in zip(idx,yt,yp)])
    result=pd.DataFrame(rows)
    stat=pd.Series({method:f1_score(df.y_true,df.y_pred,average='macro')
                    for method,df in result.groupby('method')})
    return result,stat.reindex(METHODS)

ml_test,f1_test=evaluate_ml(COLLECTION['test'])
print('Macro-F1 empat kelas pada patch uji dan akuisisi test:')
print(f1_test.round(3).to_string())

## 20. Membaca confusion matrix tanpa menyimpulkan kausalitas berlebihan

*Confusion matrix* memperlihatkan kelas mana yang saling tertukar setelah transformasi. Namun perbedaan F1 pada satu percobaan sintetik tidak otomatis menjelaskan sebabnya: dapat dipengaruhi geometri kelas, pilihan label, noise, parameter regularisasi, serta distribusi train/test. Kita akan melihat ketahanan di *stress domain* dan variasi antar-akuisisi sebelum memberi interpretasi.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(10,3.8))
for a,method in zip(ax,['Tanpa koreksi','WB + CCM']):
    df=ml_test[ml_test.method==method]
    cm=confusion_matrix(df.y_true,df.y_pred,labels=np.arange(4))
    ConfusionMatrixDisplay(cm,display_labels=CLASS_NAMES).plot(ax=a,colorbar=False,values_format='d',cmap='Blues')
    a.set_title(method);a.tick_params(axis='x',rotation=25)
plt.tight_layout();plt.show()

## 21. Pergeseran domain: perubahan iluminan dan karakteristik sensor

Skenario stress memakai matriks kamera dan distribusi iluminan yang **berbeda** dari domain train. Ini sengaja melanggar asumsi bahwa satu matriks koreksi global stabil pada semua perangkat. Jika semua metode memburuk, kesimpulannya bukan “model gagal total”, melainkan protokol kalibrasi harus mencakup domain operasi, atau diperlukan karakterisasi khusus untuk perangkat/lampu baru.

**Catatan kausal:** eksperimen stress mengubah dua faktor sekaligus (iluminan dan matriks sensor), jadi kita hanya dapat menyatakan efek gabungan, **bukan mengisolasi dampak lampu atau perangkat secara individual**.

In [ ]:
res_stress=score_collection(COLLECTION['stress'])
ml_stress,f1_stress=evaluate_ml(COLLECTION['stress'])
stress_summary=res_stress.groupby('method')['dE00'].mean().reindex(METHODS)
comparison=pd.DataFrame({'ΔE00 test':summary['mean_dE00'],'ΔE00 stress':stress_summary,
                         'macroF1 test':f1_test,'macroF1 stress':f1_stress}).reindex(METHODS)
print(comparison.round(3).to_string())
fig,ax=plt.subplots(1,2,figsize=(11,3.5))
xs=np.arange(len(METHODS));w=.36
ax[0].bar(xs-w/2,comparison['ΔE00 test'],width=w,label='Test');ax[0].bar(xs+w/2,comparison['ΔE00 stress'],width=w,label='Stress')
ax[1].bar(xs-w/2,comparison['macroF1 test'],width=w,label='Test');ax[1].bar(xs+w/2,comparison['macroF1 stress'],width=w,label='Stress')
for a in ax:a.set_xticks(xs,METHODS,rotation=15);a.legend();a.grid(alpha=.2,axis='y')
ax[0].set_ylabel('Mean ΔE00');ax[1].set_ylabel('Macro-F1');ax[1].set_ylim(0,1.05)
plt.tight_layout();plt.show()

## 22. Ketidakpastian: bootstrap pada unit akuisisi

Patch yang berasal dari satu akuisisi berbagi gain iluminan, eksposur, shading, dan keadaan sensor sehingga **tidak independen**. Karena itu, membangun interval kepercayaan dengan me-*resample* seluruh patch seolah-olah observasi independen akan mengecilkan variasi empiris.

Di sini kita hitung selisih per-akuisisi antara ΔE00 rerata **tanpa koreksi** dan **WB + CCM**, lalu lakukan bootstrap berpasangan pada ID akuisisi:

$$
\Delta_s=\overline{\Delta E}_{00,\mathrm{tanpa},s}-\overline{\Delta E}_{00,\mathrm{WB+CCM},s},\qquad
\widehat{\mathrm{CI}}_{95\%}=\left[Q_{0.025}(\overline{\Delta}^{*}),\;Q_{0.975}(\overline{\Delta}^{*})\right].
$$

Interval ini menunjukkan ketidakpastian sampling kondisional atas simulator dan kumpulan akuisisi, **bukan** uncertainty budget metrologi lengkap dan bukan jaminan cakupan dalam populasi kamera nyata.

In [ ]:
paired=res_test.groupby(['id','method'])['dE00'].mean().unstack()
difference=(paired['Tanpa koreksi']-paired['WB + CCM']).to_numpy()
rboot=np.random.default_rng(SEED+1000)
B=2500
bootstrap=np.mean(difference[rboot.integers(0,len(difference),size=(B,len(difference)))],axis=1)
lo,hi=np.quantile(bootstrap,[.025,.975])
print(f'Pengurangan mean ΔE00 berpasangan (tanpa − WB+CCM) = {difference.mean():.3f}')
print(f'95% bootstrap CI per akuisisi = [{lo:.3f}, {hi:.3f}] (n={len(difference)} akuisisi)')
fig,ax=plt.subplots(figsize=(8,3))
ax.hist(bootstrap,bins=35,alpha=.8);ax.axvline(lo,ls='--',color='k');ax.axvline(hi,ls='--',color='k')
ax.set(xlabel='Pengurangan mean ΔE00 (bootstrap)',ylabel='Jumlah resample',title='Bootstrap berpasangan pada unit akuisisi')
plt.tight_layout();plt.show()

## 23. Noise dan propagasi ketidakpastian kolorimetri

Bila data sensor dan referensi dianggap tetap, pengulangan simulasi noise membentuk distribusi estimasi warna. Untuk transformasi differentiable $\mathbf z=f(\mathbf y)$, pendekatan orde pertama menyatakan

$$
\boldsymbol\Sigma_z\approx\mathbf J_f\,\boldsymbol\Sigma_y\,\mathbf J_f^{\mathsf T},\qquad
(\mathbf J_f)_{ij}=\frac{\partial f_i}{\partial y_j}.
$$

Akan tetapi kurva sRGB, clipping, dan CIEDE2000 memuat bagian nonlinear; dekat batas gamut, pendekatan Jacobian bisa tidak memadai. Contoh Monte Carlo berikut memisahkan **variasi noise pengukuran lokal** dari ketidakpastian kalibrasi matriks, pengamatan manusia, dan kondisi lampu yang **tidak** diestimasi dengan simulasi tersebut.

In [ ]:
d=COLLECTION['test'][0]
idx=int(TEST_IDX[1])
noise_rg=np.random.default_rng(SEED+121)
repeat=1200
sample=np.clip(d['measured'][idx]+noise_rg.normal(0,.008,size=(repeat,3)),0,1)
mat=MODELS['WB + CCM']
wb_gain=REF_LIN[0]/np.maximum(d['measured'][0],.005)
repeat_lin=(sample*wb_gain)@mat
repeat_lab=rgb_to_lab_rows(lin_to_srgb(repeat_lin))
repeat_de=color.deltaE_ciede2000(repeat_lab,np.repeat(rgb_to_lab_rows(REF_SRGB[[idx]]),repeat,axis=0))
print('Patch uji',idx,'| simulasi 1200 ulang noise, ΔE00 median:',round(float(np.median(repeat_de)),3),
      '| P2.5–P97.5:',np.quantile(repeat_de,[.025,.975]).round(3))
print('Standar deviasi komponen L*,a*,b*:',np.std(repeat_lab,axis=0,ddof=1).round(3))
fig,ax=plt.subplots(1,2,figsize=(10,3.1))
ax[0].scatter(repeat_lab[:,1],repeat_lab[:,2],s=4,alpha=.12);ax[0].set(xlabel='a*',ylabel='b*',title='Sebaran koordinat Lab akibat noise')
ax[1].hist(repeat_de,bins=34);ax[1].set(xlabel='ΔE00',ylabel='Jumlah',title='Distribusi beda warna akibat noise')
plt.tight_layout();plt.show()

## 24. Clipping, eksposur, dan kegagalan model linear

Matriks koreksi memiliki kelemahan fundamental: jika satu kanal sensor mencapai maksimum, informasi variasi radiansi yang lebih tinggi hilang. Sebaliknya, respons negatif dari matriks (mungkin terjadi pada fitting tanpa pembatas) tidak dapat dipetakan langsung ke sRGB tanpa *gamut mapping* atau clipping. Oleh karena itu, laporan evaluasi warna semestinya tidak hanya menyertakan ΔE00, tetapi juga tingkat saturasi akuisisi, rentang input, dan proporsi output di luar gamut.

Pada dataset sintetis sekarang, nilai clipping output dapat berbeda antar-metode; itu **bukan** ukuran kualitas yang bisa dipakai sendiri. Penurunan clipping juga tidak otomatis berarti ΔE00 turun.

In [ ]:
fig,ax=plt.subplots(figsize=(9,3))
ax.bar(METHODS,100*summary['clipping'].to_numpy())
ax.set(ylabel='Fraksi komponen di luar [0,1] (%)',title='Audit out-of-gamut sebelum sRGB encoding')
ax.tick_params(axis='x',rotation=15)
plt.tight_layout();plt.show()
print('Jumlah nilai saturasi kamera (sebelum nilai di-clip) per domain:',
      {k:sum(d['clipped_sensor'] for d in v) for k,v in COLLECTION.items()})

## 25. Reproducibility: dataset card, split, dan pemeriksaan leakage

Sebuah eksperimen warna yang dapat direproduksi perlu menyebut sumber dan unit data, referensi yang tersedia, tujuan koreksi, versi library, random seed, cara pembagian data, serta penggunaan *ground truth*. Untuk eksperimen ini:

- **Data:** 24 patch sintetis sRGB D65, kategori yang didefinisikan penulis.
- **Sensor:** fungsi linear-plus-noise sintetis; **bukan** kamera yang diukur.
- **Unit independen utama:** akuisisi; tiap akuisisi mengandung 24 patch.
- **Train:** 18 akuisisi, 16 warna kalibrasi; **validasi:** 8 akuisisi, 4 warna berbeda; **test:** 14 akuisisi, 4 warna lain; **stress:** 8 akuisisi dengan keluarga iluminan dan matriks kamera berbeda.
- **Ground truth yang boleh digunakan di test:** nilai patch netral pertama, karena protokol mengasumsikan acuan netral diketahui saat akuisisi; nilai empat patch evaluasi uji tidak boleh dipakai untuk *fitting* atau seleksi.
- **Batas:** tidak ada pengukuran instrumen fisik, tidak ada profil ICC perangkat, dan tidak ada klaim akurasi kelas industri.

In [ ]:
assert all(set(SPLIT_SEEDS[k]).isdisjoint(set(SPLIT_SEEDS[j])) for k in SPLIT_SEEDS for j in SPLIT_SEEDS if j!=k)
assert np.intersect1d(CAL_IDX,TEST_IDX).size==0 and np.intersect1d(VAL_IDX,TEST_IDX).size==0
assert all(np.isfinite(d['measured']).all() for group in COLLECTION.values() for d in group)
assert all(d['measured'].shape==(24,3) for group in COLLECTION.values() for d in group)
assert np.allclose(srgb_to_lin(lin_to_srgb(REF_LIN)),REF_LIN,atol=1e-12)
assert abs(float(color.deltaE_ciede2000(L1,L2))-2.0425)<5e-4
print('VALIDASI: pemisahan akuisisi, warna, bentuk data, finiteness, transfer sRGB, ΔE00 lulus.')

## 26. Studi kasus: sintesis bukti dan keputusan metode

Gunakan **tiga lapis evaluasi** ketika menafsirkan hasil: (i) *color fidelity* (ΔE00 terhadap referensi), (ii) *downstream performance* (macro-F1 klasifikasi), dan (iii) keandalan di domain stress. Metode dengan ΔE00 lebih kecil tidak harus menaikkan F1 jika batas kelas berbeda dari geometri galat warna. Perbaikan pada domain test nominal juga tidak menjamin transfer ke perangkat lain.

Untuk desain eksperimen lapangan, setidaknya diperlukan: *color chart* bersertifikat atau instrumen spektrofotometer dengan kondisi pengukuran terdokumentasi, pencahayaan terkendali, RAW camera pipeline yang diketahui, kalibrasi iluminan, pengukuran berulang di hari/perangkat/batch berbeda, dan pelaporan ketidakpastian. Protokol juga perlu menyatakan apakah yang dikejar reproduksi warna untuk tampilan, stabilitas fitur klasifikasi, atau pengukuran warna bertraceability.

In [ ]:
review=comparison.copy()
review['clipping test (%)']=100*summary['clipping']
print('RINGKASAN EVALUASI (seluruh angka berasal dari dataset sintetis)')
print(review.round(3).to_string())
print('\nKeputusan harus mengikuti tujuan tugas dan domain penggunaan, bukan satu skor yang dipilih belakangan.')

## 27. Latihan analitis dan HOTS

1. **Derivasi spektral.** Dari model integral sensor, tunjukkan mengapa pengukuran RGB tidak dapat mengidentifikasi reflektansi arbitrer tanpa prior tambahan. Bangun contoh null space dan jelaskan hubungannya dengan metamerism.
2. **Kuantifikasi nonlinieritas.** Untuk dua intensitas linear 0,08 dan 0,72, hitung rata-rata linear, rata-rata encoded sRGB, dan hasil decode dari rata-rata encoded. Jelaskan kesalahan yang timbul bila gain radiometrik diterapkan di encoded sRGB.
3. **Penurunan matematis.** Turunkan solusi ridge matrix $3\times3$ menggunakan diferensiasi matriks dan terangkan kondisi ketika $\lambda>0$ memperbaiki stabilitas numerik.
4. **Kritik white balance.** Analisis mengapa target netral yang hanya berada di sudut citra bisa menghasilkan bias pada objek di sisi lain jika sensor memiliki shading spasial.
5. **Color difference.** Jelaskan mengapa $\Delta E_{ab}^*$ dan $\Delta E_{00}$ bisa mengurutkan dua metode secara berbeda. Kapan hasil ΔE00 tidak sah dibandingkan antarpenelitian?
6. **Ablation.** Ulangi pipeline dengan menghapus langkah WB dari `WB + CCM`. Bandingkan galat tiap kelas warna, bukan hanya rerata global.
7. **Kebocoran data.** Perlihatkan secara komputasional apa yang terjadi bila nilai $\lambda$ dipilih berdasarkan hasil test; jelaskan mengapa angka test yang dilaporkan kemudian optimistis.
8. **Domain shift.** Pisahkan desain simulasi sehingga hanya iluminan yang berubah dan hanya sensor yang berubah, lalu evaluasi kontribusi masing-masing terhadap hasil ΔE00 (jangan ubah dua faktor sekaligus saat menyimpulkan kausalitas).
9. **Inferensi statistik.** Uji ketahanan interval bootstrap dengan mengubah jumlah akuisisi; jelaskan mengapa bootstrap per patch berpotensi menghasilkan interval terlalu sempit.
10. **Desain metrologi.** Rancang SOP pengukuran warna produk dengan kamera smartphone: posisi chart, dark/flat field, pencahayaan, RAW/JPEG, white point, repeated measures, kalibrasi, referensi spektrofotometer, dan kriteria batas klaim.

**Tugas mini technical report (4–6 halaman):** Rumuskan hipotesis, jelaskan dataset/split, tuliskan transformasi matematis dan asumsi perangkat, laporkan minimal ΔE00, macro-F1, hasil domain shift, dan interval kepercayaan. Sertakan minimal dua failure cases dan satu usulan perbaikan yang dapat diuji. Semua angka ditarik dari notebook yang dijalankan, bukan ditulis dari contoh.

## 28. Rangkuman, koneksi antarbab, dan referensi

**Rangkuman:** RGB adalah keluaran sistem pencitraan, bukan identitas warna absolut. Pembandingan warna menuntut definisi ruang warna, titik putih, dan model akuisisi. Operasi linear radiometrik harus dilakukan dalam domain linear yang tepat. White balance diagonal, kalibrasi matriks, dan CIEDE2000 menjawab persoalan berbeda. Kesimpulan ilmiah yang baik memisahkan hasil kolorimetrik, kemampuan klasifikasi, dan generalisasi antarperangkat/iluminan; seluruh klaim terikat pada kualitas data referensi dan protokol pengujian.

**Koneksi berikutnya:** representasi warna memperluas pembahasan intensitas dan koreksi radiometrik pada notebook 01–02; filtering dan Fourier (03 dan 06) tetap penting untuk menangani noise serta struktur spasial sebelum fitur warna diekstraksi; fitur warna masuk ke **representasi fitur dan machine learning** pada notebook berikutnya. Pada tingkat lanjut, **domain shift** membahas pergantian sensor/lampu secara sistematis, sedangkan **Bayesian imaging** serta **metrologi** memperdalam ketidakpastian model dan validasi terhadap instrumen rujukan.

**Referensi terverifikasi:**

1. International Color Consortium, *sRGB registry*, IEC 61966-2-1:1999, fungsi transfer dan titik putih D65. https://registry.color.org/rgb-registry/srgb
2. scikit-image, *Color space conversion API*, `rgb2xyz`, `rgb2lab`, `deltaE_ciede2000`. https://scikit-image.org/docs/stable/api/skimage.color.html
3. Sharma, G., Wu, W., & Dalal, E. N. (2005). “The CIEDE2000 color-difference formula: Implementation notes, supplementary test data, and mathematical observations.” *Color Research & Application*, 30(1), 21–30. https://doi.org/10.1002/col.20070
4. Fairchild, M. D., *Color Appearance Models*, 3rd ed., Wiley, 2013 (rujukan teori color appearance).
5. Gonzalez, R. C. & Woods, R. E., *Digital Image Processing*, 4th ed., Pearson, 2018 (pembelajaran citra digital umum).

**Catatan hak dan interpretasi:** target warna disusun sendiri untuk pendidikan; nama ColorChecker hanya menyebut contoh target komersial sebagai konsep umum, bukan menyiratkan dataset di sini merupakan reproduksi resmi patch sertifikasi.